# 03 - Enrichment

Feature engineering sobre los DataFrames limpios de `02_clean`.

**Inventory flags:**
- Dead Stock - items con stock pero sin consumo
- Overstock - stock mayor a 12 meses de demanda
- ABC Classification - Pareto 80/15/5 sobre valor de inventario
- Days of Supply (DOS) - dias de stock restante por item
- Reorder Risk - items con menos de 30 dias de stock

**Orders flags:**
- Redundant Order - BOH ya cubre la cantidad abierta
- Open Value - valor monetario de cada orden
- Age Days - antiguedad de la orden
- Aging por proveedor - antiguedad promedio y maxima por supplier

---
## 3.1 - Cargar datos limpios

In [37]:
import pandas as pd

PATH_FILE = "../data/inventory_anon.xlsx"
sheets = pd.read_excel(PATH_FILE, sheet_name=["Inventory by Storeroom", "OPEN ORDER"])
inv = sheets["Inventory by Storeroom"]
orders = sheets["OPEN ORDER"]

In [38]:
# Limpieza (mismo proceso que 02_clean)
null_per_row = inv.isnull().sum(axis=1)
inv = inv.drop(null_per_row[null_per_row > 50].index)

null_pct = inv.isnull().sum() / len(inv)
junk_cols = null_pct[null_pct == 1.0].index.tolist() 
junk_cols.append("Unnamed: 61")
inv = inv.drop(columns=junk_cols)

null_pct_orders = orders.isnull().sum() / len(orders)
junk_cols_orders = null_pct_orders[null_pct_orders == 1.0].index.tolist()
orders = orders.drop(columns=junk_cols_orders)

print("inv:", inv.shape)
print("orders:", orders.shape)

inv: (16249, 59)
orders: (3813, 15)


---
## 3.2 - Dead Stock

Criterio: `Balance On Hand > 0` y `Average Daily Usage == 0`.

Un item con stock fisico pero sin consumo registrado es capital muerto.

In [39]:
condition = (inv["Balance On Hand"] > 0) & (inv["Average Daily Usage"] == 0)
inv["Dead Stock"] = condition.astype(int)
inv.head()

,Region Name,Country Name,Operation Name,Plant Code,Plant Name,Currency,Exchange Rate,Blanket Number,Commodity Code,Commodity,...,Status Code,Manufacturer Code,Manufacturer Name,Manufacturer Part,Total On Order,Last Modified,In ERP,Average Daily Usage,Trend,Dead Stock
0,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,MACHINE SPARES,2.0,...,A,MFG-CODE-001,MFG-001,MFG-PART-0001,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,0
1,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,A,MFG-CODE-002,MFG-002,MFG-PART-0002,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,0
2,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,A,MFG-CODE-002,MFG-002,MFG-PART-0003,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,0
3,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,A,MFG-CODE-002,MFG-002,MFG-PART-0004,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,0
4,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,A,MFG-CODE-002,MFG-002,MFG-PART-0005,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,0


In [40]:
dead = inv["Dead Stock"].sum()
dead

np.int64(3795)

In [41]:
dead_value = inv.loc[inv["Dead Stock"] == 1, "Inventory Value"].sum()
dead_value

np.float64(5651897.4939)

In [42]:
print(f"Dead stock: {dead:,} items ({dead / len(inv) * 100:.1f}%)")
print(f"Valor dead stock: ${dead_value:,.0f}")

Dead stock: 3,795 items (23.4%)
Valor dead stock: $5,651,897


---
## 3.3 - Overstock

Criterio: `Balance On Hand > Avg12Mon Usage * 12` y uso > 0.

Items con uso == 0 ya son dead stock, no overstock.

In [43]:
has_usage = inv["Avg12Mon Usage"] > 0
has_usage

0        False
1        False
2        False
3        False
4        False
         ...  
16244    False
16245    False
16246    False
16247    False
16248     True
Name: Avg12Mon Usage, Length: 16249, dtype: bool

In [44]:
# se tiene mas stock del que se necesita para 12 meses?
excess = inv["Balance On Hand"] > (inv["Avg12Mon Usage"] * 12)
excess

0        False
1        False
2        False
3        False
4        False
         ...  
16244    False
16245    False
16246    False
16247    False
16248    False
Length: 16249, dtype: bool

In [45]:
# has_usage y excess - las dos condiciones deben ser True (se usa y hay en exceso)
inv["Overstock"] = (has_usage & excess).astype(int) # convierte true/false a 1/0
inv.head()

,Region Name,Country Name,Operation Name,Plant Code,Plant Name,Currency,Exchange Rate,Blanket Number,Commodity Code,Commodity,...,Manufacturer Code,Manufacturer Name,Manufacturer Part,Total On Order,Last Modified,In ERP,Average Daily Usage,Trend,Dead Stock,Overstock
0,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,MACHINE SPARES,2.0,...,MFG-CODE-001,MFG-001,MFG-PART-0001,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,0,0
1,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,MFG-CODE-002,MFG-002,MFG-PART-0002,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,0,0
2,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,MFG-CODE-002,MFG-002,MFG-PART-0003,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,0,0
3,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,MFG-CODE-002,MFG-002,MFG-PART-0004,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,0,0
4,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,MFG-CODE-002,MFG-002,MFG-PART-0005,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,0,0


In [46]:
over = inv["Overstock"].sum()
over

np.int64(1234)

In [47]:
over_value = inv.loc[inv["Overstock"] == 1, "Inventory Value"].sum()
over_value

np.float64(1915385.0095)

In [48]:
inv.head()

,Region Name,Country Name,Operation Name,Plant Code,Plant Name,Currency,Exchange Rate,Blanket Number,Commodity Code,Commodity,...,Manufacturer Code,Manufacturer Name,Manufacturer Part,Total On Order,Last Modified,In ERP,Average Daily Usage,Trend,Dead Stock,Overstock
0,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,MACHINE SPARES,2.0,...,MFG-CODE-001,MFG-001,MFG-PART-0001,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,0,0
1,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,MFG-CODE-002,MFG-002,MFG-PART-0002,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,0,0
2,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,MFG-CODE-002,MFG-002,MFG-PART-0003,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,0,0
3,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,MFG-CODE-002,MFG-002,MFG-PART-0004,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,0,0
4,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,MFG-CODE-002,MFG-002,MFG-PART-0005,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,0,0


In [49]:
print(f"Overstock: {over:,} items ({over / len(inv) * 100:.1f}%)")
print(f"Valor overstock: ${over_value:,.0f}")

Overstock: 1,234 items (7.6%)
Valor overstock: $1,915,385


---
## 3.4 - ABC Classification

Pareto sobre `Inventory Value` acumulado:
- **A** - top 80% del valor (pocos items, alto impacto)
- **B** - siguiente 15%
- **C** - resto 5% (muchos items, bajo impacto)

In [50]:
# ordena el df por col inventory value de mayor a menor
# ascending=False = descendente (el mas caro primero)
sorted_inv = inv.sort_values("Inventory Value", ascending=False)
sorted_inv.head(3)

,Region Name,Country Name,Operation Name,Plant Code,Plant Name,Currency,Exchange Rate,Blanket Number,Commodity Code,Commodity,...,Manufacturer Code,Manufacturer Name,Manufacturer Part,Total On Order,Last Modified,In ERP,Average Daily Usage,Trend,Dead Stock,Overstock
453,North America,Mexico,OP-1,PLCODE-1,PLANT-1,MXN,1.0,PESO,MACHINE SPARES,2.0,...,MFG-CODE-028,MFG-028,MFG-PART-0388,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,1,0
15227,North America,Mexico,OP-1,PLCODE-1,PLANT-1,MXN,1.0,PESO,MACHINE SPARES,2.0,...,MFG-CODE-029,MFG-029,NaN,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,1,0
3429,North America,Mexico,OP-1,PLCODE-1,PLANT-1,MXN,1.0,PESO,MACHINE SPARES,2.0,...,MFG-CODE-010,MFG-010,MFG-PART-2292,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,1,0


In [51]:
# Como ya está ordenado de mayor a menor, el acumulado sube rápido al inicio (los items caros) y  
# se aplana al final (los baratos). 
# Eso es la base del Pareto: pocos items concentran la mayor parte del valor.
sorted_inv["Cumulative Value"] = sorted_inv["Inventory Value"].cumsum()
sorted_inv.head(3)

,Region Name,Country Name,Operation Name,Plant Code,Plant Name,Currency,Exchange Rate,Blanket Number,Commodity Code,Commodity,...,Manufacturer Name,Manufacturer Part,Total On Order,Last Modified,In ERP,Average Daily Usage,Trend,Dead Stock,Overstock,Cumulative Value
453,North America,Mexico,OP-1,PLCODE-1,PLANT-1,MXN,1.0,PESO,MACHINE SPARES,2.0,...,MFG-028,MFG-PART-0388,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,1,0,144408.95
15227,North America,Mexico,OP-1,PLCODE-1,PLANT-1,MXN,1.0,PESO,MACHINE SPARES,2.0,...,MFG-029,NaN,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,1,0,261035.24
3429,North America,Mexico,OP-1,PLCODE-1,PLANT-1,MXN,1.0,PESO,MACHINE SPARES,2.0,...,MFG-010,MFG-PART-2292,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,1,0,361768.96


In [52]:
# suma total del inventario. Es el 100% contra lo que se compara
total_value = sorted_inv["Inventory Value"].sum()

# convierte la sula acumulada a porcentaje del total
sorted_inv["Cumulative Pct"] = sorted_inv["Cumulative Value"] / total_value

sorted_inv.head(3)

,Region Name,Country Name,Operation Name,Plant Code,Plant Name,Currency,Exchange Rate,Blanket Number,Commodity Code,Commodity,...,Manufacturer Part,Total On Order,Last Modified,In ERP,Average Daily Usage,Trend,Dead Stock,Overstock,Cumulative Value,Cumulative Pct
453,North America,Mexico,OP-1,PLCODE-1,PLANT-1,MXN,1.0,PESO,MACHINE SPARES,2.0,...,MFG-PART-0388,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,1,0,144408.95,0.015782
15227,North America,Mexico,OP-1,PLCODE-1,PLANT-1,MXN,1.0,PESO,MACHINE SPARES,2.0,...,NaN,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,1,0,261035.24,0.028527
3429,North America,Mexico,OP-1,PLCODE-1,PLANT-1,MXN,1.0,PESO,MACHINE SPARES,2.0,...,MFG-PART-2292,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,1,0,361768.96,0.039536


In [53]:
sorted_inv["ABC"] = "C" # C es el valor por default

# sobreescribe con B los que acumulan hasta 95%
sorted_inv.loc[sorted_inv["Cumulative Pct"] <= 0.95, "ABC"] = "B"

# sobreescribe con "A" los que acumulan hasta 80%.
sorted_inv.loc[sorted_inv["Cumulative Pct"] <= 0.80, "ABC"] = "A"

sorted_inv.head(3)

,Region Name,Country Name,Operation Name,Plant Code,Plant Name,Currency,Exchange Rate,Blanket Number,Commodity Code,Commodity,...,Total On Order,Last Modified,In ERP,Average Daily Usage,Trend,Dead Stock,Overstock,Cumulative Value,Cumulative Pct,ABC
453,North America,Mexico,OP-1,PLCODE-1,PLANT-1,MXN,1.0,PESO,MACHINE SPARES,2.0,...,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,1,0,144408.95,0.015782,A
15227,North America,Mexico,OP-1,PLCODE-1,PLANT-1,MXN,1.0,PESO,MACHINE SPARES,2.0,...,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,1,0,261035.24,0.028527,A
3429,North America,Mexico,OP-1,PLCODE-1,PLANT-1,MXN,1.0,PESO,MACHINE SPARES,2.0,...,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,1,0,361768.96,0.039536,A


In [54]:
# copia la columan ABC del df sorted_inv de regreso al df inv
# Aunque sorted_inv esté en otro orden, pandas alinea por índice automáticamente. 
inv["ABC"] = sorted_inv["ABC"]
inv.head(3)

,Region Name,Country Name,Operation Name,Plant Code,Plant Name,Currency,Exchange Rate,Blanket Number,Commodity Code,Commodity,...,Manufacturer Name,Manufacturer Part,Total On Order,Last Modified,In ERP,Average Daily Usage,Trend,Dead Stock,Overstock,ABC
0,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,MACHINE SPARES,2.0,...,MFG-001,MFG-PART-0001,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,0,0,C
1,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,MFG-002,MFG-PART-0002,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,0,0,C
2,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,MFG-002,MFG-PART-0003,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,0,0,C


In [55]:
print(inv["ABC"].value_counts())
print()

ABC
C    13732
B     1501
A     1016
Name: count, dtype: int64



In [56]:

for cat in ["A", "B", "C"]:
    
    # filtra las filas donde ABC == cat, toma solo el "inventory value", y suma sus valores
    val = inv.loc[inv["ABC"] == cat, "Inventory Value"].sum()
    
    # crea true/false (es categoria A?) y suma los valores True, 
    # entonces sum() = total de items en esa categoria
    count = (inv["ABC"] == cat).sum()
    
    print(f"{cat}: {count:,} items - ${val:,.0f} ({val / total_value * 100:.1f}%)")

A: 1,016 items - $7,318,739 (80.0%)
B: 1,501 items - $1,373,879 (15.0%)
C: 13,732 items - $457,770 (5.0%)


In [57]:
for cat in ["A", "B", "C"]:                                                               
    count = (inv["ABC"] == cat).sum()                                                   
    val = inv.loc[inv["ABC"] == cat, "Inventory Value"].sum()
    pct_items = count / len(inv) * 100
    pct_val = val / total_value                            
    print(f"{cat}: {count:,} items ({pct_items:.1f}%) - ${val:,.0f} ({pct_val:.1f}%)") 

A: 1,016 items (6.3%) - $7,318,739 (0.8%)
B: 1,501 items (9.2%) - $1,373,879 (0.2%)
C: 13,732 items (84.5%) - $457,770 (0.1%)


- 1,016 items (6.3% del total) concentran el 80% del valor ($7.3M). Son los items A — los que más impactan financieramente.                                                         
                                                                                            
- En contraste, 13,732 items (84.5%) solo representan $457K. Son items C — muchos pero de   
  bajo valor.                                                                               
                                                                                        
**Implicación para el negocio:**                                                              
- Los items A necesitan control estricto: conteos frecuentes, proveedores confiables, reorden preciso                                                                           
- Los items C se pueden manejar con reglas simples: reorden automático, menos auditorías
- Si hay dead stock o overstock en items A, el impacto financiero es enorme               
                                                                                        
El 6% de los items controla el 80% del dinero.

---
## 3.5 - Days of Supply (DOS) (Dias de suministro)

Responde una pregunta simple: ¿para cuántos días te alcanza el stock que tienes?          

**DOS = BOH (stock actual) / Average Daily Usage (consumo diario)** 

Indica cuantos dias de stock quedan al ritmo de consumo actual.

Items con uso == 0 obtienen DOS = infinito (no se consumen, el stock nunca se agota).

In [58]:
import numpy as np

inv["DOS"] = np.where(
    inv["Average Daily Usage"] > 0, # el item de consume?
    inv["Balance On Hand"] / inv["Average Daily Usage"], # calcula los dias de stock
    np.inf # si no se consume, el stock nunca se acaba
)

In [59]:
inv.head(3)

,Region Name,Country Name,Operation Name,Plant Code,Plant Name,Currency,Exchange Rate,Blanket Number,Commodity Code,Commodity,...,Manufacturer Part,Total On Order,Last Modified,In ERP,Average Daily Usage,Trend,Dead Stock,Overstock,ABC,DOS
0,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,MACHINE SPARES,2.0,...,MFG-PART-0001,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,0,0,C,inf
1,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,MFG-PART-0002,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,0,0,C,inf
2,North America,Mexico,OP-1,PLCODE-1,PLANT-1,EUR,1.0,EURO,PERISHABLE,3.0,...,MFG-PART-0003,0.0,2019-02-19 01:09:17.087,N,0.0,No Trend,0,0,C,inf


In [60]:
# filtra solo items que tienen DOS diferente de infinito (los que si se consumen)
# filtrar la columna DOS para ocultar los que dicen infinito, y te quedas con los demás.
finite_dos = inv.loc[inv["DOS"] != np.inf, "DOS"] 

print(f"DOS promedio (items con uso): {finite_dos.mean():.0f} dias")
print(f"DOS mediana: {finite_dos.median():.0f} dias")
print(f"Items con DOS infinito (sin uso): {(inv['DOS'] == np.inf).sum():,}")

DOS promedio (items con uso): 769 dias
DOS mediana: 48 dias
Items con DOS infinito (sin uso): 11,774


- 11,774 items (72%) no se consumen DOS infinito. Son la mayoría del inventario y están ahí sin moverse.

- Promedio vs mediana: 769 vs 48 días — la diferencia es enorme. Significa que hay items con DOS de miles de días que jalan el promedio hacia arriba. La mediana (48 días) es más realista: la mitad de los items activos tienen menos de 2 meses de stock.

- 48 días de mediana — para items que sí se usan, el stock dura mes y medio. Es razonable para una planta industrial, pero hay que cruzar con la clasificación ABC: si un item A tiene DOS de 5 días, es urgente reordenar.
                                                                                            
- En resumen: el inventario tiene un problema de dos caras - la mayoría no se mueve (dead stock) y los que sí se mueven tienen stock para aproximadamente 48 días.

---
## 3.6 - Reorder Risk

Criterio: `Average Daily Usage > 0` y `BOH < Average Daily Usage * 30`.

Items activos con menos de 30 dias de stock. Riesgo de desabasto si no se reordena.

In [61]:
active = inv["Average Daily Usage"] > 0
low_stock = inv["Balance On Hand"] < (inv["Average Daily Usage"] * 30)
inv["Reorder Risk"] = (active & low_stock).astype(int)

In [62]:
risk = inv["Reorder Risk"].sum()
risk_value = inv.loc[inv["Reorder Risk"] == 1, "Inventory Value"].sum()
print(f"Reorder risk: {risk:,} items ({risk / len(inv) * 100:.1f}%)")
print(f"Valor en riesgo: ${risk_value:,.0f}")

Reorder risk: 1,871 items (11.5%)
Valor en riesgo: $390,324


- 1,871 items (11.5%) tienen menos de 30 días de stock y sí se consumen. Si no se reordenan 
  pronto, habrá desabasto.                                                                  
                                                                                            
- Pero el valor es relativamente bajo: $390K comparado con los $9.1M del inventario total (4.3%). Esto sugiere que la mayoría de items en riesgo son categoría C (baratos). 



---
## 3.7 - Enrich Orders

Columnas nuevas en Open Orders:
- `Open Value` - Open Qty * Price 
  - cuánto dinero representa cada orden. Cantidad pedida multiplicada por precio unitario


- `Redundant Order` - 1 si BOH >= Open Qty (ya se tiene stock suficiente)
  - ¿la orden es innecesaria? Si ya tienes en almacén (BOH) igual o más de lo que se pidio,¿para qué la pediste? Marca 1 si es redundante, 0 si no. 


- `Age Days` - dias desde Issue Date hasta la orden mas reciente
  - cuántos días lleva abierta la orden. Se calcula desde la fecha de emisión (Issue Date) hasta la fecha más reciente del dataset.

In [ ]:
# cuanto dinero representa cada orden, cantidad pedida x precio unitario
orders["Open Value"] = orders["Open Qty"] * orders["Price"]

# si balance on hand >= cantidad pedida = entonces ya se tiene stock suficiente (redundante)
orders["Redundant Order"] = (orders["BOH"] >= orders["Open Qty"]).astype(int) # de true/false a 1/0

orders.head(3)

,Plant Code,Item Number,Item Description,UOM,PO Number,Issue Date,Order Qty,Open Qty,Required Date,BOH,Price,Supplier Code,Supplier Name,Last Modified,Vendor PO,Open Value,Redundant Order,Age Days
0,PLCODE-2,ITEM-8207,ITEM-DESC-8185,EA,NaN,2017-07-06 16:03:24.780,1,1,2017-07-06,38.0,32.59,SUP-CODE-001,SUPPLIER-001,2019-02-19 01:09:04.533,VPO-0001,32.59,1,592
1,PLCODE-2,ITEM-5094,ITEM-DESC-4443,EA,NaN,2018-10-11 15:33:29.090,41,41,2018-10-18,134.0,6.48,SUP-CODE-002,SUPPLIER-002,2019-02-19 01:09:04.533,VPO-0002,265.68,1,130
2,PLCODE-2,ITEM-0862,ITEM-DESC-0844,EA,NaN,2017-12-13 14:55:05.830,1,1,2017-12-20,0.0,10666.67,SUP-CODE-003,SUPPLIER-003,2019-02-19 01:09:04.533,VPO-0003,10666.67,0,432


In [66]:

# convierte la columna Issue Date a formato datetime para poder hacer operaciones con fechas
orders["Issue Date"] = pd.to_datetime(orders["Issue Date"], errors="coerce")

# la fecha mas reciente del dataset - se usa como referencia para calcular antiguedad
last_date = orders["Issue Date"].max()

# cuantos dias lleva abierta cada orden desde su emision hasta la fecha mas reciente
# last_date - Issue Date = diferencia en tiempo, .dt.days lo convierte a numero de dias
orders["Age Days"] = (last_date - orders["Issue Date"]).dt.days

orders.head(3)

,Plant Code,Item Number,Item Description,UOM,PO Number,Issue Date,Order Qty,Open Qty,Required Date,BOH,Price,Supplier Code,Supplier Name,Last Modified,Vendor PO,Open Value,Redundant Order,Age Days
0,PLCODE-2,ITEM-8207,ITEM-DESC-8185,EA,NaN,2017-07-06 16:03:24.780,1,1,2017-07-06,38.0,32.59,SUP-CODE-001,SUPPLIER-001,2019-02-19 01:09:04.533,VPO-0001,32.59,1,592
1,PLCODE-2,ITEM-5094,ITEM-DESC-4443,EA,NaN,2018-10-11 15:33:29.090,41,41,2018-10-18,134.0,6.48,SUP-CODE-002,SUPPLIER-002,2019-02-19 01:09:04.533,VPO-0002,265.68,1,130
2,PLCODE-2,ITEM-0862,ITEM-DESC-0844,EA,NaN,2017-12-13 14:55:05.830,1,1,2017-12-20,0.0,10666.67,SUP-CODE-003,SUPPLIER-003,2019-02-19 01:09:04.533,VPO-0003,10666.67,0,432


In [68]:
# cuantas ordenes son redundantes (sum de 1s en la columna)
redundant = orders["Redundant Order"].sum()                                               
                                                                                            
# valor total en dolares de las ordenes redundantes que son igual a 1                                     
redundant_value = orders.loc[orders["Redundant Order"] == 1, "Open Value"].sum()  

orders.head(3)

,Plant Code,Item Number,Item Description,UOM,PO Number,Issue Date,Order Qty,Open Qty,Required Date,BOH,Price,Supplier Code,Supplier Name,Last Modified,Vendor PO,Open Value,Redundant Order,Age Days
0,PLCODE-2,ITEM-8207,ITEM-DESC-8185,EA,NaN,2017-07-06 16:03:24.780,1,1,2017-07-06,38.0,32.59,SUP-CODE-001,SUPPLIER-001,2019-02-19 01:09:04.533,VPO-0001,32.59,1,592
1,PLCODE-2,ITEM-5094,ITEM-DESC-4443,EA,NaN,2018-10-11 15:33:29.090,41,41,2018-10-18,134.0,6.48,SUP-CODE-002,SUPPLIER-002,2019-02-19 01:09:04.533,VPO-0002,265.68,1,130
2,PLCODE-2,ITEM-0862,ITEM-DESC-0844,EA,NaN,2017-12-13 14:55:05.830,1,1,2017-12-20,0.0,10666.67,SUP-CODE-003,SUPPLIER-003,2019-02-19 01:09:04.533,VPO-0003,10666.67,0,432


In [69]:
# cantidad, porcentaje, valor y ordenes viejas (mas de 1 anio)
print(f"Redundant orders: {redundant:,} ({redundant / len(orders) * 100:.1f}%)")
print(f"Valor redundante: ${redundant_value:,.0f}")
print(f"Ordenes > 365 dias: {(orders['Age Days'] > 365).sum():,}")

Redundant orders: 2,538 (66.6%)
Valor redundante: $2,556,553
Ordenes > 365 dias: 209


- **66.6% de las órdenes son redundantes — 2 de cada 3 órdenes piden material que ya se tiene en almacén. Son $2.5M en compras innecesarias.**

- 209 órdenes llevan más de 1 año abiertas — nadie las cerró ni las canceló. Probablemente se olvidaron o el proveedor nunca entregó.                                               

**¿Qué significa para el negocio?**                                                           
                                                            
- **El proceso de compras no revisa el stock antes de pedir**. Se está ordenando material que ya está en el estante. Si se cancelaran las órdenes redundantes, se liberarían $2.5M en compromisos de compra.                                                                    
                                                            
- Las órdenes viejas (>365 días) indican falta de seguimiento - **nadie revisa las órdenes abiertas periódicamente para cerrarlas o escalarlas con el proveedor.**

---
## 3.8 - Aging (edad de las ordenes) por proveedor

¿qué tan viejas son las órdenes de cada supplier? - cuandos dias se lleva esperando?

Antiguedad promedio y maxima de ordenes abiertas agrupadas por supplier.

  - Un proveedor con promedio alto = es lento entregando en general                         
  - Un proveedor con max alto = tiene órdenes olvidadas o abandonadas
  - Un proveedor con muchas órdenes + valor alto = es prioritario resolver porque representa
   más dinero estancado 

Revisar que suppliers deben mercancia y desde cuando?

In [72]:
# agrupa las ordenes por proveedor y calcula por antiguedad
#   count = cuantas ordenes tiene ese proveedor
#   mean = antiguedad promedio en dias
#   max = la orden mas vieja de ese proveedor
#   sum = valor total de ordenes abiertas
aging = orders.groupby("Supplier Name").agg(
    orders_count=("Age Days", "count"),
    avg_age_days=("Age Days", "mean"),
    max_age_days=("Age Days", "max"),
    total_open_value=("Open Value", "sum")
).round(0).sort_values("total_open_value", ascending=False)

#orders.head(3)
aging

,orders_count,avg_age_days,max_age_days,total_open_value
Supplier Name,,,,
SUPPLIER-001,835,139.0,711,1110774.0
SUPPLIER-026,33,145.0,403,831704.0
SUPPLIER-004,42,304.0,410,708639.0
SUPPLIER-013,428,56.0,641,442951.0
SUPPLIER-010,253,97.0,707,331200.0
...,...,...,...,...
SUPPLIER-157,1,20.0,20,85.0
SUPPLIER-054,1,269.0,269,52.0
SUPPLIER-154,1,21.0,21,29.0


In [73]:
print(f"Proveedores unicos: {len(aging)}")
print(f"\nTop 10 por valor abierto:")
aging.head(10)

Proveedores unicos: 190

Top 10 por valor abierto:


,orders_count,avg_age_days,max_age_days,total_open_value
Supplier Name,,,,
SUPPLIER-001,835,139.0,711,1110774.0
SUPPLIER-026,33,145.0,403,831704.0
SUPPLIER-004,42,304.0,410,708639.0
SUPPLIER-013,428,56.0,641,442951.0
SUPPLIER-010,253,97.0,707,331200.0
SUPPLIER-005,162,209.0,979,252992.0
SUPPLIER-059,242,37.0,259,251878.0
SUPPLIER-120,3,30.0,83,190864.0
SUPPLIER-021,39,86.0,466,183528.0


---
## 3.9 - Resumen de columnas generadas

In [78]:
# resumen de todas las columnas nuevas que se crearon en este notebook

# columnas agregadas al inventario
new_inv_cols = ["Dead Stock", "Overstock", "ABC", "DOS", "Reorder Risk"]
print("\n=== Inventory - columnas nuevas ===")
# describe() estadisticas basicas: count, mean, std, min, max, percentiles
inv[new_inv_cols].describe()


=== Inventory - columnas nuevas ===


/home/judah/411013/a2-c1-oraElLabora/projects/inventory-report-generator/venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4596: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a
/home/judah/411013/a2-c1-oraElLabora/projects/inventory-report-generator/venv/lib/python3.12/site-packages/pandas/core/nanops.py:1020: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,Dead Stock,Overstock,DOS,Reorder Risk
count,16249.000000,16249.000000,16249.0,16249.000000
mean,0.233553,0.075943,inf,0.115146
std,0.423104,0.264915,NaN,0.319207
min,0.000000,0.000000,0.0,0.000000
25%,0.000000,0.000000,1095.0,0.000000
50%,0.000000,0.000000,NaN,0.000000
75%,0.000000,0.000000,NaN,0.000000
max,1.000000,1.000000,inf,1.000000


In [76]:
# columnas agregadas a las ordenes
new_order_cols = ["Open Value", "Redundant Order", "Age Days"]
print("\n=== Orders - columnas nuevas ===")
orders[new_order_cols].describe()


=== Orders - columnas nuevas ===


,Open Value,Redundant Order,Age Days
count,3813.000000,3813.000000,3813.000000
mean,2070.915898,0.665618,97.270653
std,9804.081791,0.471836,133.778019
min,0.000000,0.000000,0.000000
25%,190.000000,0.000000,18.000000
50%,500.600000,1.000000,33.000000
75%,1449.200000,1.000000,123.000000
max,334235.000000,1.000000,1139.000000


In [77]:

# shapes finales — cuantas filas x columnas tiene cada df despues del enrichment
print(f"\nInventory final: {inv.shape}")
print(f"Orders final: {orders.shape}")


Inventory final: (16249, 64)
Orders final: (3813, 18)


---
## Conclusion

El enrichment revelo 4 problemas principales en el inventario:

1. **Dead stock masivo** — 3,795 items (23.4%) con stock pero sin consumo. Representan $5.6M (61.8% del valor total). Es capital muerto ocupando espacio en almacen.

2. **Ordenes redundantes** — 2,538 ordenes (66.6%) piden material que ya se tiene en almacen. Son $2.5M en compromisos de compra innecesarios. El proceso de compras no revisa el stock antes de ordenar.

3. **Overstock** — 1,234 items tienen stock para mas de 12 meses de consumo. $1.9M amarrados en exceso de inventario.

4. **Riesgo de desabasto** — 1,871 items activos tienen menos de 30 dias de stock. Aunque su valor es bajo ($390K), si son criticos para produccion, un faltante puede detener la linea.

**Hallazgo adicional:** 209 ordenes llevan mas de 1 anio abiertas sin que nadie las cierre o escale con el proveedor. SUPPLIER-005 tiene ordenes de hasta 979 dias (~2.7 anios).

**Columnas generadas:**

| DataFrame | Columna | Tipo | Descripcion |
|-----------|---------|------|-------------|
| Inventory | Dead Stock | flag (0/1) | BOH > 0 y uso diario == 0 |
| Inventory | Overstock | flag (0/1) | BOH > 12 meses de consumo |
| Inventory | ABC | categoria (A/B/C) | Pareto 80/15/5 sobre valor |
| Inventory | DOS | numerico | Dias de stock restante |
| Inventory | Reorder Risk | flag (0/1) | Menos de 30 dias de stock |
| Orders | Open Value | numerico ($) | Open Qty * Price |
| Orders | Redundant Order | flag (0/1) | BOH >= Open Qty |
| Orders | Age Days | numerico (dias) | Dias desde emision |

